# Evaluation: Austrian Tax Law LLM Comparison

This notebook evaluates three LLM approaches on the Austrian Tax Law dataset:
- **Inference**: Llama 3.3 70B via Groq API (zero-shot)
- **Fine-tuning**: Llama 3.1 8B fine-tuned with QLoRA
- **RAG**: FAISS + paraphrase-multilingual-MiniLM-L12-v2

**Metrics:**
- **ROUGE-L**: measures longest common subsequence overlap between generated and reference answers
- **BERTScore**: measures semantic similarity using multilingual BERT embeddings

Reference answers come from the shared course dataset (`correct_answer` column).

**Note on ROUGE-L:** Model answers are longer and more detailed than the short reference answers. ROUGE-L penalizes verbosity even when the answer is correct. BERTScore is therefore a more appropriate metric for this task.

**Note on the dataset:** The shared dataset contains a known ID issue in the VAT-INTL category, where all 82 entries share the ID `VAT-INTL-001`. This is a data quality issue in the shared dataset, not in the model outputs.

## Step 1: Install dependencies

In [13]:
pip install pandas rouge-score bert-score


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## Step 2: Configuration

Place this notebook in the same folder as the four CSV files and the paths will work as-is.

In [18]:
GT_PATH        = "Austrian Tax Law LLM Prompt Assignment - Dataset.csv"
INFERENCE_PATH = "/Users/edwarddimapilis/Desktop/Uni/sbwl/Data Science/Kurs 4/project/12309335_Dimapilis_Edward/evaluation/inference_submission.csv"
FT_PATH        = "/Users/edwarddimapilis/Desktop/Uni/sbwl/Data Science/Kurs 4/project/12309335_Dimapilis_Edward/evaluation/ft_submission.csv"
RAG_PATH       = "/Users/edwarddimapilis/Desktop/Uni/sbwl/Data Science/Kurs 4/project/12309335_Dimapilis_Edward/evaluation/rag_submission.csv"

## Step 3: Load data

In [19]:
import pandas as pd

gt     = pd.read_csv(GT_PATH)[['id', 'correct_answer']]
inf_df = pd.read_csv(INFERENCE_PATH)
ft_df  = pd.read_csv(FT_PATH)
rag_df = pd.read_csv(RAG_PATH)

inf_merged = gt.merge(inf_df.rename(columns={'answer': 'inference'}), on='id', how='inner')
ft_merged  = gt.merge(ft_df.rename(columns={'answer': 'ft'}),        on='id', how='inner')
rag_merged = gt.merge(rag_df.rename(columns={'answer': 'rag'}),      on='id', how='inner')

print(f"Inference rows: {len(inf_merged)}")
print(f"Fine-tuning rows: {len(ft_merged)}")
print(f"RAG rows: {len(rag_merged)}")

Inference rows: 643
Fine-tuning rows: 643
RAG rows: 643


## Step 4: Clean answers

In [20]:
def clean(series):
    return series.fillna('').astype(str)

ref_inf = clean(inf_merged['correct_answer']).tolist()
hyp_inf = clean(inf_merged['inference']).tolist()

ref_ft  = clean(ft_merged['correct_answer']).tolist()
hyp_ft  = clean(ft_merged['ft']).tolist()

ref_rag = clean(rag_merged['correct_answer']).tolist()
hyp_rag = clean(rag_merged['rag']).tolist()

print("Data ready.")

Data ready.


## Step 5: Compute ROUGE-L

In [21]:
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=False)

def compute_rouge_l(hypotheses, references):
    scores = [scorer.score(ref, hyp)['rougeL'].fmeasure
              for hyp, ref in zip(hypotheses, references)]
    return sum(scores) / len(scores)

rouge_inf = compute_rouge_l(hyp_inf, ref_inf)
rouge_ft  = compute_rouge_l(hyp_ft,  ref_ft)
rouge_rag = compute_rouge_l(hyp_rag, ref_rag)

print(f"ROUGE-L  Inference:   {rouge_inf:.4f}")
print(f"ROUGE-L  Fine-tuning: {rouge_ft:.4f}")
print(f"ROUGE-L  RAG:         {rouge_rag:.4f}")

ROUGE-L  Inference:   0.1796
ROUGE-L  Fine-tuning: 0.1141
ROUGE-L  RAG:         0.0652


## Step 6: Compute BERTScore

BERTScore uses `bert-base-multilingual-cased` to measure semantic similarity. The model is downloaded on first run (~700MB) and computation takes a few minutes on CPU.

In [23]:
from bert_score import score as bert_score_fn

def compute_bertscore(hypotheses, references):
    P, R, F1 = bert_score_fn(
        hypotheses,
        references,
        model_type='bert-base-multilingual-cased',
        lang='de',
        verbose=False
    )
    return F1.mean().item()

print("Computing BERTScore for Inference...")
bert_inf = compute_bertscore(hyp_inf, ref_inf)

print("Computing BERTScore for Fine-tuning...")
bert_ft  = compute_bertscore(hyp_ft, ref_ft)

print("Computing BERTScore for RAG...")
hyp_rag = [h[:512] if h else "keine Antwort" for h in hyp_rag]
bert_rag = compute_bertscore(hyp_rag, ref_rag)

print(f"\nBERTScore (F1)  Inference:   {bert_inf:.4f}")
print(f"BERTScore (F1)  Fine-tuning: {bert_ft:.4f}")
print(f"BERTScore (F1)  RAG:         {bert_rag:.4f}")

Computing BERTScore for Inference...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1448.90it/s]
BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Computing BERTScore for Fine-tuning...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 12326.53it/s]
BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Computing BERTScore for RAG...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 15860.65it/s]
BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



BERTScore (F1)  Inference:   0.7093
BERTScore (F1)  Fine-tuning: 0.6674
BERTScore (F1)  RAG:         0.6157


## Step 7: Results table

In [24]:
results = pd.DataFrame({
    'Model':           ['Inference (Llama 3.3 70B)', 'Fine-tuning (Llama 3.1 8B)', 'RAG'],
    'ROUGE-L':         [rouge_inf, rouge_ft, rouge_rag],
    'BERTScore (F1)':  [bert_inf,  bert_ft,  bert_rag]
})

results['ROUGE-L']        = results['ROUGE-L'].map('{:.4f}'.format)
results['BERTScore (F1)'] = results['BERTScore (F1)'].map('{:.4f}'.format)

print(results.to_string(index=False))

                     Model ROUGE-L BERTScore (F1)
 Inference (Llama 3.3 70B)  0.1796         0.7093
Fine-tuning (Llama 3.1 8B)  0.1141         0.6674
                       RAG  0.0652         0.6157


## Step 8: Error analysis

We look at the 5 worst-performing examples per model according to ROUGE-L to understand what kinds of errors each model makes.

In [25]:
gt_full = pd.read_csv(GT_PATH)[['id', 'prompt']]

def get_per_sample_rouge(hypotheses, references):
    return [scorer.score(ref, hyp)['rougeL'].fmeasure
            for hyp, ref in zip(hypotheses, references)]

inf_merged['rouge_l'] = get_per_sample_rouge(hyp_inf, ref_inf)
ft_merged['rouge_l']  = get_per_sample_rouge(hyp_ft,  ref_ft)
rag_merged['rouge_l'] = get_per_sample_rouge(hyp_rag, ref_rag)

def show_worst(merged_df, answer_col, model_name, n=5):
    worst = merged_df.nsmallest(n, 'rouge_l').merge(gt_full, on='id', how='left')
    print(f"\n=== {model_name}: {n} worst answers by ROUGE-L ===")
    for _, row in worst.iterrows():
        print(f"\nID: {row['id']}")
        print(f"Question:  {str(row['prompt'])[:120]}")
        print(f"Reference: {str(row['correct_answer'])[:200]}")
        print(f"Generated: {str(row[answer_col])[:200]}")
        print(f"ROUGE-L:   {row['rouge_l']:.4f}")
        print("-" * 60)

show_worst(inf_merged, 'inference', 'Inference')
show_worst(ft_merged,  'ft',        'Fine-tuning')
show_worst(rag_merged, 'rag',       'RAG')


=== Inference: 5 worst answers by ROUGE-L ===

ID: SELF-006
Question:  Was regelt das Gemeinnützigkeitsreformgesetz 2023 hinsichtlich Spenden?
Reference: Es erweitert den Kreis der spendenbegünstigten Einrichtungen erheblich (z.B. auf Bildung).
Generated: Das Gemeinnützigkeitsreformgesetz 2023 regelt unter anderem die Spendenabsetzbarkeit. Laut § 4a EStG (Einkommensteuergesetz) können Spenden an bestimmte, vom Finanzamt anerkannte, gemeinnützige Organi
ROUGE-L:   0.0000
------------------------------------------------------------

ID: SELF-040
Question:  Was ist der Gewinnfreibetrag nach § 10 EStG?
Reference: Eine steuerliche Begünstigung für natürliche Personen mit betrieblichen Einkünften (Grundfreibetrag + investitionsbedingter GFB).
Generated: Der Gewinnfreibetrag nach § 10 EStG (Einkommensteuergesetz) in Österreich beträgt 13% des Gewinns, jedoch höchstens 45.350 Euro (Stand 2023). Dieser Betrag ist steuerfrei und reduziert somit die steue
ROUGE-L:   0.0000
----------------------

## Step 9: Save results

In [26]:
results.to_csv('evaluation_results.csv', index=False)
print("Saved to evaluation_results.csv")

Saved to evaluation_results.csv
